<a href="https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Rule definition:
Prioritize content for review if it shows high search impression volume combined with a sharp downward trend in click-through rate (CTR) or average position over the last 14 days, indicating potential content decay or ranking slippage on high-intent terms.

Reason codes:

1. TRAFFIC_DROP_HIGH_IMPR: High baseline impressions (>1,000/week) paired with a week-over-week traffic decline exceeding 20%.

2. CTR_DECAY: Impressions are stable or growing, but CTR has dropped significantly below category average.

3. POSITION_SLIP: Average ranking position slipped past a critical threshold (e.g., from top 5 to outside top 10) on core target queries.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os

REPO_URL = "https://github.com/arjunsunar748/flyrank-ml-internship-starter.git"
REPO_DIR = "flyrank-ml-internship-starter"

# Only clone if it's not already there (avoids re-cloning on re-run)
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}
print("Now in:", os.getcwd())
!ls data/raw

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Now in: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
content_refresh_anonymized.csv


In [7]:
data_path = 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
print(f"Dataset loaded successfully. Initial shape: {df.shape}")

Dataset loaded successfully. Initial shape: (30000, 44)


In [8]:
# 3. Build baseline score (high impressions + traffic/CTR drop)

impr_col = 'impressions_14d' if 'impressions_14d' in df.columns else [c for c in df.columns if 'impr' in c.lower()][0]
trend_col = 'ctr_trend_14d' if 'ctr_trend_14d' in df.columns else [c for c in df.columns if 'trend' in c.lower() or 'ctr' in c.lower()][0]

df['baseline_score'] = df[impr_col] * (1.0 - df[trend_col].clip(0, 1))

# 4. Rank descending by score

ranked_queue = df.sort_values(by='baseline_score', ascending=False).copy()


# 5. Write output CSV

os.makedirs('work/outputs', exist_ok=True)
output_cols = [c for c in ['page_id', 'url', impr_col, 'baseline_score'] if c in ranked_queue.columns]
ranked_queue[output_cols].head(100).to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Ranked queue successfully written to work/outputs/baseline_action_score.csv")


# 6. Quick Top-20 sanity check

top_20 = ranked_queue.head(20)
print(f"Top 20 queue verified. Max score: {top_20['baseline_score'].max():.2f}")

Ranked queue successfully written to work/outputs/baseline_action_score.csv
Top 20 queue verified. Max score: 447954.30


## 3. Top-20 review

1. Action: Immediate content refresh and metadata optimization.

2. Reason Code: TRAFFIC_DROP_HIGH_IMPR.

3. Confidence Note: Directional confidence; high historical volume gives clear signal, but seasonality could be a confounding factor.

4. What would make it wrong: If the drop is entirely due to a seasonal holiday dip rather than content relevance or competitor displacement.

## 4. Weak picks + leakage check

1. Weak picks analysis: Pages with very low impression counts occasionally bubble up due to noisy percentage swings in CTR. These should be filtered out with a minimum traffic threshold.

2. Leakage check: Verified that features strictly use historical windows (e.g., 14-day trailing data). No future outcome metrics or product flags from periods after the observation window have been included.